In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions
import pandas as pd

In [3]:
def add_label_based_on_id(id_value):

    if "non-antioxidant" in id_value:
        return 0
    else:
        return 1

In [4]:
path_export = "../../processed_dataset/"
path_input = "../../raw_dataset/"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "Thanh-Lam et al"

- Read doc and labels

In [5]:
df_data = pd.read_excel(f"{path_input}/{name_source}/data.xlsx")
df_data["label"] = df_data["SEQCLASS"].apply(add_label_based_on_id)
df_data = df_data.rename(columns={'SEQUENCE':'sequence'})
df_data = df_data[['sequence', 'label']]


- Checking duplicates

In [6]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape
#no duplicates

((0, 0), (0, 0), (466, 2))

- Checking labels

In [7]:
df_data["label"].value_counts()

label
0    392
1     74
Name: count, dtype: int64

- Working with metadata


In [8]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
52,data.xlsx,Thanh-Lam et al,Dataset,Static,Creative Commons BY 4.0,No,2020,2020-10-06,2026-04-07,xlsx,Sequence,Enzyme/protein classification,Antioxidant,Obtained from other databases,"Sampling from Swiss-Prot, Sampling from UniProt",https://www.mdpi.com/2079-7737/9/10/325/s1,https://pmc.ncbi.nlm.nih.gov/articles/PMC7599600/,No information


In [9]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'data.xlsx',
 'name source': 'Thanh-Lam et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons BY 4.0',
 'reports constant updates': 'No',
 'year of publication': 2020,
 'last update date': Timestamp('2020-10-06 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Obtained from other databases',
 'obtaining positive dataset': 'Sampling from Swiss-Prot, Sampling from UniProt',
 'repository or server': 'https://www.mdpi.com/2079-7737/9/10/325/s1',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7599600/',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-16 18:31:36'}

In [10]:
df_data["label"] = df_data["label"].astype(int)

In [11]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = df_data.shape[0]
dict_metadata['positive_examples'] = df_data[df_data["label"] == 1].shape[0]
dict_metadata['negative_examples'] = df_data[df_data["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': 'data.xlsx',
 'name source': 'Thanh-Lam et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons BY 4.0',
 'reports constant updates': 'No',
 'year of publication': 2020,
 'last update date': Timestamp('2020-10-06 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Obtained from other databases',
 'obtaining positive dataset': 'Sampling from Swiss-Prot, Sampling from UniProt',
 'repository or server': 'https://www.mdpi.com/2079-7737/9/10/325/s1',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7599600/',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-16 18:31:36',
 'number_of_records': 466,
 'number_of_collected_sequences': 466,
 'number_of_unique_sequences': 466,
 'positive_examples': 74,
 'negative_

- Export data

In [12]:
UtilsFunctions.make_directory(f"{path_export}{name_task}/{name_source}")

In [13]:
UtilsFunctions.export_json(f"{path_export}{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)

In [14]:
df_data.to_csv(f"{path_export}{name_task}/{name_source}/processed_data.csv", index=False)